In [2]:
### mini project
#### Building a Rag system with Langchain and ChromaDB


In [3]:
import os
os.makedirs("DataSet/PDF_files",exist_ok=True)

In [4]:
# Method 2: PyMuPDFLoader (Fast and accurate)
print("\n3️⃣ PyMuPDFLoader")
try:
    pymupdf_loader = PyMuPDFLoader("C:/Users/Himanshugarse/Rangkush/PDF/PDF_files")
    pymupdf_docs = pymupdf_loader.load()
    
    print(f"  Loaded {len(pymupdf_docs)} pages")
    print(f"  Includes detailed metadata")
    print(pymupdf_docs)
except Exception as e:
    print(f"  Error: {e}")


3️⃣ PyMuPDFLoader
  Error: name 'PyMuPDFLoader' is not defined


In [5]:

from langchain_community.document_loaders import DirectoryLoader,TextLoader

In [6]:
# Method 2: PyMuPDFLoader (Fast and accurate)
print("\n3️⃣ PyMyPDFLoader")
try:
    pymupdf_loader = PyMyPDFLoader("C:/Users/Himanshugarse/Rangkush/PDF/PDF_files")
    pymupdf_docs = pymupdf_loader.load()
    
    print(f"  Loaded {len(pymupdf_docs)} pages")
    print(f"  Includes detailed metadata")
    print(pymupdf_docs)
except Exception as e:
    print(f"  Error: {e}")


3️⃣ PyMyPDFLoader
  Error: name 'PyMyPDFLoader' is not defined


In [7]:
# --- Import necessary libraries ---
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
import os

# --- Step 1: Define your directory path ---
pdf_directory = "C:/Users/Himanshugarse/Rangkush/PDF/PDF_files"

# --- Step 2: Use DirectoryLoader to load all PDFs using PyMuPDFLoader ---
print("\n📄 Loading all PDF files using PyMuPDFLoader...\n")

try:
    pdf_loader = DirectoryLoader(
        path=pdf_directory,
        glob="*.pdf",               # Loads only PDF files
        loader_cls=PyMuPDFLoader,
        show_progress=True
    )

    # --- Step 3: Load all documents (each page = one Document) ---
    pdf_docs = pdf_loader.load()
    print(f"✅ Loaded {len(pdf_docs)} pages from all PDFs in '{pdf_directory}'\n")

    # --- Step 4: Group documents by file for better previews ---
    # Create a dictionary to group pages by their PDF source file
    pdf_files = {}
    for doc in pdf_docs:
        file_path = doc.metadata.get("source", "Unknown file")
        pdf_files.setdefault(file_path, []).append(doc)

    # --- Step 5: Preview each PDF file ---
    print("📘 --- PDF File Previews ---\n")
    for file_path, docs in pdf_files.items():
        file_name = os.path.basename(file_path)
        print(f"📄 File: {file_name}")
        print(f"   Total pages loaded: {len(docs)}")

        # Preview first few lines of the first page
        first_page_text = docs[0].page_content.strip().replace("\n", " ")
        preview_text = first_page_text[:400] + ("..." if len(first_page_text) > 400 else "")
        print(f"   📝 Preview (first 400 chars): {preview_text}")
        print("-" * 100)

except Exception as e:
    print(f"❌ Error while loading PDFs: {e}")


📄 Loading all PDF files using PyMuPDFLoader...



100%|██████████| 1/1 [00:00<00:00,  4.09it/s]

✅ Loaded 4 pages from all PDFs in 'C:/Users/Himanshugarse/Rangkush/PDF/PDF_files'

📘 --- PDF File Previews ---

📄 File: Hitler_Historical_Analysis_Book.pdf
   Total pages loaded: 4
   📝 Preview (first 400 chars): The Rise and Fall of Adolf Hitler: A Historical Examination Written by Himanshu Garse This book provides a factual, historical, and analytical account of Adolf Hitler, his ideology, his role in World War II, and the global consequences of his actions. It does not glorify or support any extremist ideology; instead, it focuses strictly on historical events and their impact on the world. CHAPTER 1: E...
----------------------------------------------------------------------------------------------------


In [8]:
import os
import re
import unicodedata
from typing import List
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

# -----------------------------
# SmartPDFProcessor Class
# -----------------------------
class SmartPDFProcessor:
    def __init__(self, chunk_size=1000, chunk_overlap=100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[" ", "\n"]
        )

    def process_pdf(self, pdf_path: str) -> List[Document]:
        loader = PyMuPDFLoader(pdf_path)
        pages = loader.load()
        processed_chunks = []

        for page_num, page in enumerate(pages):
            cleaned_text = self._clean_text(page.page_content)
            if len(cleaned_text.strip()) < 50:
                continue
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text),
                    "source_file": os.path.basename(pdf_path)
                }]
            )
            processed_chunks.extend(chunks)

        print(f"✅ Processed {len(processed_chunks)} chunks from '{os.path.basename(pdf_path)}'")
        return processed_chunks

    def process_directory(self, folder_path: str) -> List[Document]:
        all_chunks = []
        pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".pdf")]
        if not pdf_files:
            print("⚠️ No PDF files found in the directory.")
            return []

        print(f"\n📁 Found {len(pdf_files)} PDFs in '{folder_path}'\n")
        for pdf_file in pdf_files:
            pdf_path = os.path.join(folder_path, pdf_file)
            chunks = self.process_pdf(pdf_path)
            if chunks:
                print(f"\n📘 Preview of '{pdf_file}':")
                print(chunks[0].page_content[:300], "...\n")
            all_chunks.extend(chunks)

        print(f"\n✅ Total Chunks Created from Directory: {len(all_chunks)}")
        return all_chunks

    def _clean_text(self, text: str) -> str:
        text = unicodedata.normalize("NFKC", text)
        text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
        text = text.replace("’", "'").replace("‘", "'")
        text = text.replace("“", '"').replace("”", '"')
        text = text.replace("—", "-").replace("–", "-")
        text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

In [9]:
import os
import re
import unicodedata
from typing import List
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

# -----------------------------
# SmartPDFProcessor Class
# -----------------------------
class SmartPDFProcessor:
    def __init__(self, chunk_size=1000, chunk_overlap=100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[" ", "\n"]
        )

    def process_pdf(self, pdf_path: str) -> List[Document]:
        loader = PyMuPDFLoader(pdf_path)
        pages = loader.load()
        processed_chunks = []

        for page_num, page in enumerate(pages):
            cleaned_text = self._clean_text(page.page_content)
            if len(cleaned_text.strip()) < 50:
                continue
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text),
                    "source_file": os.path.basename(pdf_path)
                }]
            )
            processed_chunks.extend(chunks)

        print(f"✅ Processed {len(processed_chunks)} chunks from '{os.path.basename(pdf_path)}'")
        return processed_chunks

    def process_directory(self, folder_path: str) -> List[Document]:
        all_chunks = []
        pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".pdf")]
        if not pdf_files:
            print("⚠️ No PDF files found in the directory.")
            return []

        print(f"\n📁 Found {len(pdf_files)} PDFs in '{folder_path}'\n")
        for pdf_file in pdf_files:
            pdf_path = os.path.join(folder_path, pdf_file)
            chunks = self.process_pdf(pdf_path)
            if chunks:
                print(f"\n📘 Preview of '{pdf_file}':")
                print(chunks[0].page_content[:300], "...\n")
            all_chunks.extend(chunks)

        print(f"\n✅ Total Chunks Created from Directory: {len(all_chunks)}")
        return all_chunks

    def _clean_text(self, text: str) -> str:
        text = unicodedata.normalize("NFKC", text)
        text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
        text = text.replace("’", "'").replace("‘", "'")
        text = text.replace("“", '"').replace("”", '"')
        text = text.replace("—", "-").replace("–", "-")
        text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

In [10]:
folder_path = "C:/Users/Himanshugarse/Rangkush/PDF/PDF_files"

processor = SmartPDFProcessor(chunk_size=400, chunk_overlap=20)
all_chunks = processor.process_directory(folder_path)

# Preview first 2 chunks
for i, chunk in enumerate(all_chunks[:2]):
    print(f"\n📄 Chunk {i+1} (Page {chunk.metadata['page']} of {chunk.metadata['total_pages']}):")
    print(chunk.page_content[:300], "...")


📁 Found 1 PDFs in 'C:/Users/Himanshugarse/Rangkush/PDF/PDF_files'

✅ Processed 14 chunks from 'Hitler_Historical_Analysis_Book.pdf'

📘 Preview of 'Hitler_Historical_Analysis_Book.pdf':
The Rise and Fall of Adolf Hitler: A Historical Examination Written by Himanshu Garse This book provides a factual, historical, and analytical account of Adolf Hitler, his ideology, his role in World War II, and the global consequences of his actions. It does not glorify or support any extremist ide ...


✅ Total Chunks Created from Directory: 14

📄 Chunk 1 (Page 1 of 4):
The Rise and Fall of Adolf Hitler: A Historical Examination Written by Himanshu Garse This book provides a factual, historical, and analytical account of Adolf Hitler, his ideology, his role in World War II, and the global consequences of his actions. It does not glorify or support any extremist ide ...

📄 Chunk 2 (Page 1 of 4):
world. CHAPTER 1: Early Life and Rise to Power Adolf Hitler was born on April 20, 1889, in Braunau am Inn, A

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "sentence-transformers/all-MiniLM-L6-v2"
print(f"⚙️ Loading Hugging Face embeddings model: {model_name}")
embeddings = HuggingFaceEmbeddings(model_name=model_name)
print("✅ Model loaded successfully.")

⚙️ Loading Hugging Face embeddings model: sentence-transformers/all-MiniLM-L6-v2
✅ Model loaded successfully.


In [12]:
texts = [chunk.page_content for chunk in all_chunks]

print("🔢 Generating embeddings for all chunks...")
chunk_embeddings = embeddings.embed_documents(texts)

print(f"✅ Generated embeddings for {len(chunk_embeddings)} chunks")
print(f"Each embedding vector has {len(chunk_embeddings[0])} dimensions")

# Preview first embedding vector (first 10 values)
print("\n🔹 Example embedding (first 10 values):")
print(chunk_embeddings[0][:10])

🔢 Generating embeddings for all chunks...
✅ Generated embeddings for 14 chunks
Each embedding vector has 384 dimensions

🔹 Example embedding (first 10 values):
[0.004919893573969603, 0.07240107655525208, -0.06628889590501785, -0.016246862709522247, 0.04188736528158188, 0.09743858128786087, 0.026708532124757767, 0.1269540637731552, -0.0423426479101181, -0.020175665616989136]


In [13]:
# ...existing code...
# ensure these variables/modules exist: `embeddings` (HuggingFaceEmbeddings instance) and `all_chunks`
persist_directory = "chroma_db"

from langchain.vectorstores import Chroma

# use the embeddings object's query function as embedding_function
# (if your embeddings variable is different, use the appropriate callable)
vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings.embed_query)

query = "Who is the author of Introduction to Algorithms"
results = vectorstore.similarity_search(query, k=3)

print(f"\n🔍 Top 3 search results for query: '{query}'")
for i, r in enumerate(results):
    print(f"\nResult {i+1} (Page {r.metadata.get('page','?')}, File: {r.metadata.get('source_file','?')}):")
    print(r.page_content[:500], "...")
# ...existing code...

C:\Users\Himanshugarse\AppData\Local\Temp\ipykernel_7432\400248708.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings.embed_query)


ImportError: Could not import chromadb python package. Please install it with `pip install chromadb`.

In [14]:
import os
import chromadb
from langchain.vectorstores import Chroma
from typing import List, Any

def save_chunks_to_chroma(all_chunks: List[Any], embeddings, persist_directory: str = "chroma_db"):
    """
    Persist a list of chunk Documents to a Chroma DB.
    Call from your notebook after `all_chunks` and `embeddings` are available:
      from save_chroma import save_chunks_to_chroma
      save_chunks_to_chroma(all_chunks, embeddings)
    """
    if not all_chunks:
        print("No chunks to store")
        return None

    texts = [c.page_content for c in all_chunks]
    metadatas = [c.metadata for c in all_chunks]

    os.makedirs(persist_directory, exist_ok=True)

    vectorstore = Chroma.from_texts(
        texts=texts,
        embedding=embeddings,
        metadatas=metadatas,
        persist_directory=persist_directory,
    )

    vectorstore.persist()
    print(f"✅ Stored {len(texts)} chunks in Chroma DB at '{persist_directory}'")
    return vectorstore

if __name__ == "__main__":
    print("This module provides save_chunks_to_chroma(all_chunks, embeddings). Import and call it from your notebook.")

This module provides save_chunks_to_chroma(all_chunks, embeddings). Import and call it from your notebook.
